In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [22]:
df = pd.read_csv("../data/raw_dataset.csv")
df.head()

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Person ID                374 non-null    int64  
 1   Gender                   374 non-null    object 
 2   Age                      374 non-null    int64  
 3   Occupation               374 non-null    object 
 4   Sleep Duration           374 non-null    float64
 5   Quality of Sleep         374 non-null    int64  
 6   Physical Activity Level  374 non-null    int64  
 7   Stress Level             374 non-null    int64  
 8   BMI Category             374 non-null    object 
 9   Blood Pressure           374 non-null    object 
 10  Heart Rate               374 non-null    int64  
 11  Daily Steps              374 non-null    int64  
 12  Sleep Disorder           155 non-null    object 
dtypes: float64(1), int64(7), object(5)
memory usage: 38.1+ KB


In [24]:
df.describe()

,Person ID,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps
count,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000,374.000000
mean,187.500000,42.184492,7.132086,7.312834,59.171123,5.385027,70.165775,6816.844920
std,108.108742,8.673133,0.795657,1.196956,20.830804,1.774526,4.135676,1617.915679
min,1.000000,27.000000,5.800000,4.000000,30.000000,3.000000,65.000000,3000.000000
25%,94.250000,35.250000,6.400000,6.000000,45.000000,4.000000,68.000000,5600.000000
50%,187.500000,43.000000,7.200000,7.000000,60.000000,5.000000,70.000000,7000.000000
75%,280.750000,50.000000,7.800000,8.000000,75.000000,7.000000,72.000000,8000.000000
max,374.000000,59.000000,8.500000,9.000000,90.000000,8.000000,86.000000,10000.000000


In [25]:
print(df['Occupation'].unique())

print(df['BMI Category'].unique())

print(df['Sleep Disorder'].unique())

['Software Engineer' 'Doctor' 'Sales Representative' 'Teacher' 'Nurse'
 'Engineer' 'Accountant' 'Scientist' 'Lawyer' 'Salesperson' 'Manager']
['Overweight' 'Normal' 'Obese' 'Normal Weight']
[nan 'Sleep Apnea' 'Insomnia']


#### Dataset Columns:

Person ID: An identifier for each individual. (int)

Gender: The gender of the person (Male/Female). (str)

Age: The age of the person in years. (int)

Occupation: The occupation or profession of the person.(str)

Sleep Duration (hours): The number of hours the person sleeps per day. ()

Quality of Sleep (scale: 1-10): A subjective rating of the quality of sleep, ranging from 1 to 10.

Physical Activity Level (minutes/day): The number of minutes the person engages in physical activity daily.

Stress Level (scale: 1-10): A subjective rating of the stress level experienced by the person, ranging from 1 to 10.

BMI Category: The BMI category of the person (e.g., Underweight, Normal, Overweight).

Blood Pressure (systolic/diastolic): The blood pressure measurement of the person, indicated as systolic pressure over diastolic pressure.

Heart Rate (bpm): The resting heart rate of the person in beats per minute.

Daily Steps: The number of steps the person takes per day.

Sleep Disorder: The presence or absence of a sleep disorder in the person (None, Insomnia, Sleep Apnea).


### Columns to predict
Quality of Sleep (scale: 1-10): A subjective rating of the quality of sleep, ranging from 1 to 10.


# Pre-processing

In [26]:
# 1. Supprimer Person ID (pas utile pour la prediction)
df = df.drop('Person ID', axis=1)

# 2. Traiter Blood Pressure - separer en 2 colonnes numeriques
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True).astype(int)
df = df.drop('Blood Pressure', axis=1)

# 3. Remplir les valeurs manquantes pour Sleep Disorder
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

# 4. Separer features (X) et target (y)
X = df.drop('Quality of Sleep', axis=1)
y = df['Quality of Sleep']

# if quality of sleep is less than 7 put 0 else 1
y = y.apply(lambda x: 0 if x < 8 else 1)

# print balance of the dataset
print(y.value_counts())

# 5. Identifier les colonnes categoriques et numeriques
categorical_features = ['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']
numerical_features = ['Age', 'Sleep Duration', 'Physical Activity Level', 'Stress Level',
                      'Heart Rate', 'Daily Steps', 'Systolic_BP', 'Diastolic_BP']


# 6. Split train/test avant preprocessing pour eviter data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# 7. Preprocessing avec ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])



Quality of Sleep
0    194
1    180
Name: count, dtype: int64

Train set size: (299, 12)
Test set size: (75, 12)


# Définition du problème :

L'objectif est de prédire la colonne "Quality of Sleep". Ce problème pourrait être abordé comme une tâche de régression, mais pour cette première exploration (et afin de simplifier l'analyse), nous l'avons transformé en un problème de classification binaire.

# Test d'un modele

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Créer un pipeline avec preprocessing et modèle de base
basic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

# Entraîner le modèle
basic_model.fit(X_train, y_train)

# Prédire sur le jeu de test
y_pred = basic_model.predict(X_test)

# Afficher l'accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy du modèle basique : {accuracy:.4f}")

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
print("\nMatrice de confusion :")
print(cm)

# Rapport de classification: précision, rappel, f1-score...
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))


Accuracy du modèle basique : 1.0000

Matrice de confusion :
[[45  0]
 [ 0 30]]

Rapport de classification :
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        45
           1       1.00      1.00      1.00        30

    accuracy                           1.00        75
   macro avg       1.00      1.00      1.00        75
weighted avg       1.00      1.00      1.00        75



## Conclusion

Les résultats obtenus montrent que notre modèle de classification atteint une performance parfaite sur les données test, avec une précision de 100% et une matrice de confusion sans erreurs. Cela suggère que la tâche de classification binaire pour prédire la "Quality of Sleep" dans ce contexte est relativement simple à résoudre, probablement en raison d'une forte séparation des classes ou d'une structure de données très favorable.

Pour aller plus loin et afin de proposer un défi plus intéressant et pertinent dans le cadre du projet final, nous choisirons d'aborder ce problème sous l'angle de la régression. Cela permettra d'estimer la qualité du sommeil sous forme continue, rendant la tâche plus riche et offrant des perspectives d'analyse et de modélisation plus approfondies.
